In [1]:
import pandas as pd
import numpy as np

In [2]:
ratings = pd.read_csv(
    r"C:\Users\user\Downloads\ml-100k\ml-100k\u.data",
    sep="\t",
    names=["user_id" , "movie_id", "rating" , "timestamp"]
)

In [3]:
print(ratings.head(10))

   user_id  movie_id  rating  timestamp
0      196       242       3  881250949
1      186       302       3  891717742
2       22       377       1  878887116
3      244        51       2  880606923
4      166       346       1  886397596
5      298       474       4  884182806
6      115       265       2  881171488
7      253       465       5  891628467
8      305       451       3  886324817
9        6        86       3  883603013


In [4]:
print(ratings.shape)

(100000, 4)


In [5]:
ratings["user_id"].nunique()

943

In [6]:
ratings["movie_id"].nunique()

1682

In [7]:
print(ratings["rating"].value_counts().sort_index())

rating
1     6110
2    11370
3    27145
4    34174
5    21201
Name: count, dtype: int64


In [8]:
rating_matrix = ratings.pivot(index="user_id",columns="movie_id",values="rating")

In [9]:
small_matrix=rating_matrix.iloc[:5, :10].copy()

In [10]:
print(small_matrix)

movie_id   1    2    3    4    5    6    7    8    9    10
user_id                                                   
1         5.0  3.0  4.0  3.0  3.0  5.0  4.0  1.0  5.0  3.0
2         4.0  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  2.0
3         NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN
4         NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN
5         4.0  3.0  NaN  NaN  NaN  NaN  NaN  NaN  NaN  NaN


In [11]:
yArray=np.array(small_matrix.copy())
nan_mask = np.isnan(yArray[:,:])
yArray[nan_mask] = 0
print(yArray)

[[5. 3. 4. 3. 3. 5. 4. 1. 5. 3.]
 [4. 0. 0. 0. 0. 0. 0. 0. 0. 2.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [4. 3. 0. 0. 0. 0. 0. 0. 0. 0.]]


In [12]:
rArray=np.array(small_matrix.copy())
rArray[nan_mask] = 0
rArray[~nan_mask] = 1
print(rArray)

[[1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
 [1. 0. 0. 0. 0. 0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [1. 1. 0. 0. 0. 0. 0. 0. 0. 0.]]


In [13]:
def costFunc(W,X,B_film,B_user):
    user_num ,w_features_num  = W.shape
    film_num, x_features_num = X.shape
    result = np.zeros((user_num,film_num))
    
    for j in range(user_num):
        for i in range(film_num):
            if(rArray[j,i] == 1):
                result[j,i] = np.square(np.sum(np.dot(X[i],W[j])) + B_film[i] + B_user[j] - Y[j,i]) / 2
    return np.sum(result)

In [20]:
def costFunc(W,X,B_film,B_user):
    error = W @ X.T + B_user[:,np.newaxis] + B_film[np.newaxis,:] - yArray
    cost = rArray * error
    cost = np.sum(np.square(cost)) / 2
    return cost

In [19]:
def predict(W,X,B_film,B_user):
    prediction_matrix = W @ X.T + B_user[:,np.newaxis] + B_film[np.newaxis,:]
    return prediction_matrix

In [39]:
np.array(
    [
    [1,4,5],
    [2,6,7],
    [3,8,9]
    ]).sum(axis=1)

array([10, 15, 20])

In [48]:
def gradientDescent(W, X, B_film, B_user, iteration, learning_rate, momentum = 0.9):
    
    dW = None; dX = None; dB_film = None; dB_user = None;
    
    vW = np.zeros_like(W)
    vX = np.zeros_like(X)
    vB_film = np.zeros_like(B_film)
    vB_user = np.zeros_like(B_user)
    
    
    for i in range(iteration):
        
        prediction = W @ X.T + B_user[:,np.newaxis] + B_film[np.newaxis,:]
        error = (prediction - yArray) * rArray
        
        dW = np.dot(error,X)
        dX = np.dot(error,W.T)
        dB_film = np.sum(error, axis = 0)
        dB_user = np.sum(error, axis = 1)

        vW = momentum * vW + learning_rate * dW
        vX = momentum * vX + learning_rate * dX
        vB_film = momentum * vB_film + learning_rate * dB_film
        vB_user = momentum * vB_user + learning_rate * dB_user
        
        W = W - vW
        X = X - vX
        B_film = dB_film - vB_film
        B_user = dB_user - vB_user

        if i % 100 == 0:
            print(i,"th iteration     Loss:", np.sum( np.square(error) / 2))

    return W, X, B_film, B_user